# Vault Authentication via Teleport SVID

This notebook demonstrates authenticating to HashiCorp Vault using an **X.509 SVID issued by Teleport Workload Identity** — no static tokens, passwords, or secrets stored anywhere.

Normally, getting a workload into Vault means putting *something* long-lived in its way — a static token, a userpass credential, or a network proxy that vouches for it. Here, Vault trusts Teleport's SPIFFE certificate authority directly. Teleport mints a short-lived X.509 SVID for the workload, the workload presents that SVID to Vault over mutual TLS, and Vault's **cert auth method** validates it against the trusted CA — no intermediary needed. The identity *is* the certificate, and it's only valid for an hour.

The flow, which you'll build step by step in this notebook:

1. Teleport issues a short-lived X.509 SVID to the workload
2. The workload presents the SVID to Vault via mutual TLS
3. Vault verifies the certificate against the trusted Teleport SPIFFE CA
4. Vault issues a token scoped by policy

**No Teleport network proxy is in the path** — Vault validates the Teleport CA directly.

Each step below has code you run plus a short explanation of *why* it's there. Read both — a couple of these steps (the `common_name` field, the `--type=tls-spiffe` export flag) fail in confusing ways if skipped, and the explanation tells you why before you hit the error.

## How It Works

```
Workload (this notebook)
  │  tsh workload-identity issue-x509 --name-selector vault-test
  ↓
Teleport
  │  Issues X.509 SVID signed by Teleport SPIFFE CA
  │  URI SAN: spiffe://<cluster>/vault-test
  ↓
vault login -method=cert -client-cert=svid.pem -client-key=svid_key.pem
  ↓
Vault cert auth
  │  Verifies cert against registered Teleport SPIFFE CA
  │  Checks URI SAN matches allowed_uri_sans glob
  │  Issues Vault token with attached policy
  ↓
Secret read
```

That diagram is four separate pieces of trust chained together, and Steps 3–7 below build each one in order:

- **Teleport's SPIFFE CA** — a certificate authority Teleport maintains separately from its host and user CAs, dedicated to signing workload identities (Step 4).
- **A `WorkloadIdentity` resource** — defines the SPIFFE ID path this workload is allowed to claim, and the rules that gate who can claim it (Step 3).
- **An issued X.509 SVID** — a short-lived certificate minted from that CA on the workload's behalf (Step 6).
- **Vault's `cert` auth method** — configured to trust the SPIFFE CA and accept only certificates whose SPIFFE ID matches a defined pattern (Step 5).

### App Access Proxy vs. Workload Identity

Registering Vault as a Teleport Application and proxying `tsh apps login` traffic through it is a good fit for human SSO access to the Vault UI — Teleport controls who can reach Vault at all, and Vault still authenticates the proxied connection with its own native methods. This notebook does the opposite: it skips the proxy and has Vault validate the Teleport CA directly at the TLS layer, which is the better fit when the thing authenticating is a workload rather than a person.

| | App Access Proxy | Workload Identity (this guide) |
|---|---|---|
| **Network path** | Teleport proxies to Vault | Direct connection |
| **Auth method** | Vault native (token, userpass…) | Vault cert auth via SPIFFE mTLS |
| **Audit log** | Teleport logs HTTP requests | Vault audit log + Teleport logs SVID issuance |
| **Best for** | Human SSO access to Vault UI/CLI | Machine-to-machine workload auth |

## Prerequisites

- `tsh` and `tctl` (Teleport 18+)
- Active `tsh login` session with admin or equivalent access
- `vault` CLI installed and reachable
- `openssl` (for SVID verification)
- Python 3.11+

Validated with Teleport 18.8.3, HashiCorp Vault 2.0.3 (OSS), and `tsh` 18.8.3 — none of this requires
an Enterprise or Cloud edition of either product.

> **No Vault yet?** See the [Appendix](#appendix-vault-docker-setup) at the bottom of this notebook
> for a complete Docker Compose setup with TLS that matches this demo.

## Step 1: Install Dependencies

In [ ]:
import sys
!{sys.executable} -m pip install python-dotenv --quiet

## Step 2: Configure Environment

Copy `.env.example` to `.env` and fill in your values before running this cell.

```
TELEPORT_CLUSTER=yourcluster.teleport.sh
TELEPORT_USER=your-user@example.com
VAULT_ADDR=https://your-vault-host:8200
VAULT_TOKEN=<root-or-admin-token>   # only needed for Vault setup cells
VAULT_CACERT=                       # path to vault CA cert; leave empty to skip TLS verify (dev only)
```

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(dotenv_path='.env', override=True)

TELEPORT_CLUSTER = os.environ.get('TELEPORT_CLUSTER', '')
TELEPORT_USER    = os.environ.get('TELEPORT_USER', '')
VAULT_ADDR       = os.environ.get('VAULT_ADDR', '')
VAULT_TOKEN      = os.environ.get('VAULT_TOKEN', '')
VAULT_CACERT     = os.environ.get('VAULT_CACERT', '')

for name, val in [
    ('TELEPORT_CLUSTER', TELEPORT_CLUSTER),
    ('TELEPORT_USER',    TELEPORT_USER),
    ('VAULT_ADDR',       VAULT_ADDR),
]:
    if not val:
        raise ValueError(f'Set {name} in .env')

# All generated files (SVIDs, YAMLs, CA cert) go here — gitignored
WORK_DIR = Path('./workspace')
WORK_DIR.mkdir(exist_ok=True)

SVID_DIR       = WORK_DIR / 'svid'
SVID_CERT      = str(SVID_DIR / 'svid.pem')
SVID_KEY       = str(SVID_DIR / 'svid_key.pem')
SPIFFE_CA_FILE = str(WORK_DIR / 'teleport-spiffe-ca.pem')

SPIFFE_URI_GLOB  = f'spiffe://{TELEPORT_CLUSTER}/*'
VAULT_TLS_VERIFY = VAULT_CACERT if VAULT_CACERT else False

print(f'Cluster        : {TELEPORT_CLUSTER}')
print(f'User           : {TELEPORT_USER}')
print(f'Vault address  : {VAULT_ADDR}')
print(f'SPIFFE URI glob: {SPIFFE_URI_GLOB}')
print(f'TLS verify     : {VAULT_TLS_VERIFY}')
print(f'Working dir    : {WORK_DIR.resolve()}')
if not VAULT_CACERT:
    print('⚠  VAULT_CACERT not set — using verify=False (dev only, not for production)')


## Step 3: Teleport RBAC Setup

Two resources are required:

1. **WorkloadIdentity** — defines the SPIFFE ID and the issuance rules
2. **Issuer role** — grants your user permission to issue workload identity tokens

The RBAC model here is label-based: the issuer role needs a `workload_identity_labels` matcher
that matches the labels on the `WorkloadIdentity` resource before it can issue from it, plus a
`workload_identity_token` `use` rule to actually mint the SVID.

The `common_name` field in the WorkloadIdentity is critical: Vault uses it as the alias name in cert auth.
Omitting it causes a cryptic `missing name in alias` error at login time.

In [ ]:
workload_identity_yaml = f"""kind: workload_identity
version: v1
metadata:
  name: vault-test
  labels:
    app: vault-test
spec:
  spiffe:
    id: /vault-test
    x509:
      subject_template:
        common_name: vault-test
  rules:
    allow:
      - conditions:
          - attribute: user.name
            eq:
              value: {TELEPORT_USER}
"""

outfile = WORK_DIR / 'workload-identity.yaml'
outfile.write_text(workload_identity_yaml)
print(workload_identity_yaml)


In [ ]:
import subprocess

result = subprocess.run(
    ['tctl', 'create', '-f', str(WORK_DIR / 'workload-identity.yaml')],
    capture_output=True, text=True,
)
# Exit code 1 with 'already exists' is not a failure
print(result.stdout or result.stderr)
if result.returncode not in (0, 1):
    raise RuntimeError(f'tctl create failed: {result.stderr}')


In [ ]:
issuer_role_yaml = """kind: role
version: v8
metadata:
  name: workload-identity-issuer
spec:
  allow:
    workload_identity_labels:
      "*": ["*"]
    rules:
      - resources: [workload_identity]
        verbs: [list, read]
      - resources: [workload_identity_token]
        verbs: [use]
"""

outfile = WORK_DIR / 'workload-identity-issuer-role.yaml'
outfile.write_text(issuer_role_yaml)
print(issuer_role_yaml)


In [ ]:
import json

# Create the role
result = subprocess.run(
    ['tctl', 'create', '-f', str(WORK_DIR / 'workload-identity-issuer-role.yaml')],
    capture_output=True, text=True,
)
print(result.stdout or result.stderr)

# Fetch current user roles so we can append without clobbering existing assignments
result = subprocess.run(
    ['tctl', 'get', f'users/{TELEPORT_USER}', '--format=json'],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print(f'Could not fetch user: {result.stderr}')
    print(f'Run manually: tctl users update {TELEPORT_USER} --set-roles <current-roles>,workload-identity-issuer')
else:
    users = json.loads(result.stdout)
    current_roles = users[0]['spec']['roles']
    if 'workload-identity-issuer' in current_roles:
        print(f'Role already assigned. Current roles: {current_roles}')
    else:
        new_roles = current_roles + ['workload-identity-issuer']
        r2 = subprocess.run(
            ['tctl', 'users', 'update', TELEPORT_USER, '--set-roles', ','.join(new_roles)],
            capture_output=True, text=True,
        )
        print(r2.stdout or r2.stderr)
        print(f'Updated roles: {new_roles}')

print()
print('Re-login to pick up the new role:')
print(f'  tsh logout --all && tsh login --proxy={TELEPORT_CLUSTER}')

## Step 4: Export the Teleport SPIFFE CA

Vault needs the Teleport SPIFFE CA to verify SVIDs. Use `--type=tls-spiffe` — **not** `tls-host`.
The host CA signs SSH certificates; the SPIFFE CA signs workload X.509 certificates.
Using the wrong CA causes a `failed to match all constraints` error in Vault.

In [ ]:
result = subprocess.run(
    ['tctl', 'auth', 'export', '--type=tls-spiffe'],
    capture_output=True, text=True,
)
if result.returncode != 0:
    raise RuntimeError(f'tctl auth export failed: {result.stderr}')

Path(SPIFFE_CA_FILE).write_text(result.stdout)
print(f'Wrote {SPIFFE_CA_FILE}  ({len(result.stdout)} bytes)')

In [ ]:
result = subprocess.run(
    ['openssl', 'x509', '-in', SPIFFE_CA_FILE, '-noout', '-subject', '-issuer', '-dates'],
    capture_output=True, text=True,
)
print(result.stdout)

## Step 5: Configure Vault

Five sub-steps, each in its own cell below. Run them in order.

| Sub-step | What it does |
|---|---|
| **5a** | Build the admin environment dict used by every `vault` CLI call |
| **5b** | Enable the `cert` auth method in Vault |
| **5c** | Register the Teleport SPIFFE CA and bind it to a policy |
| **5d** | Create the `app-secrets` policy scoping secret access |
| **5e** | Write a test secret so there is something to read in Step 7 |

> **Requires an active Vault root or admin token in `VAULT_TOKEN`.**

### 5a: Build the Vault Admin Environment

Constructs the environment dict passed to every `vault` CLI subprocess in this step.
Sets `VAULT_ADDR`, `VAULT_TOKEN`, and either `VAULT_CACERT` or `VAULT_SKIP_VERIFY=true`.

**Expected output:** `Vault environment configured  (VAULT_SKIP_VERIFY=true)`

In [ ]:
if not VAULT_TOKEN:
    raise ValueError('Set VAULT_TOKEN in .env for Vault setup steps')

# Start from the current environment, then clear any inherited Vault TLS vars
# so they cannot conflict with what we set explicitly below.
vault_env = {**os.environ, "VAULT_ADDR": VAULT_ADDR, "VAULT_TOKEN": VAULT_TOKEN}
vault_env.pop("VAULT_CACERT", None)
vault_env.pop("VAULT_SKIP_VERIFY", None)

if VAULT_CACERT:
    vault_env["VAULT_CACERT"] = VAULT_CACERT
else:
    vault_env["VAULT_SKIP_VERIFY"] = "true"

tls_mode = f"VAULT_CACERT={VAULT_CACERT}" if VAULT_CACERT else "VAULT_SKIP_VERIFY=true"
print(f"Vault environment configured  ({tls_mode})")


### 5b: Enable the Cert Auth Method

Enables Vault's built-in `cert` auth method at `auth/cert/`. Only needs to run once —
re-running when it is already enabled prints a warning but does not fail.

**Expected output:** `Success! Enabled cert auth method at: cert/`

In [ ]:
result = subprocess.run(
    ['vault', 'auth', 'enable', 'cert'],
    capture_output=True, text=True, env=vault_env,
)
# 'path is already in use' (exit 2) is not a failure
print(result.stdout or result.stderr)

### 5c: Register the Teleport SPIFFE CA

Uploads the exported Teleport SPIFFE CA and creates a cert role named `teleport-svid`.
Any SVID whose URI SAN matches `allowed_uri_sans` and is signed by this CA will be
granted the `app-secrets` policy.

**Expected output:** `Success! Data written to: auth/cert/certs/teleport-svid`

In [ ]:
# Register the Teleport SPIFFE CA and restrict which SPIFFE IDs can authenticate.
# Narrow the glob in production: spiffe://<cluster>/vault-test
result = subprocess.run(
    [
        'vault', 'write', 'auth/cert/certs/teleport-svid',
        f'certificate=@{SPIFFE_CA_FILE}',
        f'allowed_uri_sans={SPIFFE_URI_GLOB}',
        'policies=app-secrets',
        'ttl=1h',
    ],
    capture_output=True, text=True, env=vault_env,
)
if result.returncode != 0:
    raise RuntimeError(f'vault write failed: {result.stderr}')
print(result.stdout)

### 5d: Create the App Secrets Policy

Writes the `app-secrets` HCL policy that grants `read` access to `secret/data/myapp/*`.
This is the policy attached to tokens issued after a successful SVID login.

**Expected output:** `Success! Uploaded policy: app-secrets`

In [ ]:
policy_hcl = """\
path "secret/data/myapp/*" {
  capabilities = ["read"]
}
"""

result = subprocess.run(
    ['vault', 'policy', 'write', 'app-secrets', '-'],
    input=policy_hcl,
    capture_output=True, text=True, env=vault_env,
)
if result.returncode != 0:
    raise RuntimeError(f'vault policy write failed: {result.stderr}')
print(result.stdout)

### 5e: Write a Test Secret

Puts a placeholder secret at `secret/myapp/config` so Step 7 has something to read.
Skip this cell if your Vault already has data at that path.

**Expected output:** `created_time`, `version`, and other KV v2 metadata

In [ ]:
# Write a test secret so there is something to read in the demo
result = subprocess.run(
    ['vault', 'kv', 'put', '-mount=secret', 'myapp/config',
     'db_host=db.internal', 'api_key=demo-secret-value'],
    capture_output=True, text=True, env=vault_env,
)
if result.returncode != 0:
    raise RuntimeError(f'vault kv put failed: {result.stderr}')
print(result.stdout)

## Step 6: Issue an X.509 SVID

`tsh workload-identity issue-x509` issues a short-lived SVID signed by the Teleport SPIFFE CA.
The `--name-selector` value matches the `metadata.name` of the WorkloadIdentity created in Step 3.

Files written to `./svid/`:

| File | Purpose |
|------|--------|
| `svid.pem` | X.509 SVID certificate |
| `svid_key.pem` | Private key |
| `svid_bundle.pem` | SPIFFE trust bundle (CA chain) |

**Requires an active `tsh login` session with the `workload-identity-issuer` role.**

In [ ]:
os.makedirs(SVID_DIR, exist_ok=True)

result = subprocess.run(
    ['tsh', 'workload-identity', 'issue-x509',
     '--output', SVID_DIR,
     '--name-selector', 'vault-test'],
    capture_output=True, text=True,
)
if result.returncode != 0:
    raise RuntimeError(f'tsh workload-identity issue-x509 failed: {result.stderr}')

print(result.stdout or 'SVID issued successfully')
print()
for f in sorted(Path(SVID_DIR).iterdir()):
    print(f'  {f}  ({f.stat().st_size} bytes)')

In [ ]:
# Verify SVID before authenticating — confirm SPIFFE URI SAN, subject, and expiry
result = subprocess.run(
    ['openssl', 'x509', '-in', SVID_CERT, '-noout',
     '-subject', '-issuer', '-dates', '-ext', 'subjectAltName'],
    capture_output=True, text=True,
)
print(result.stdout)

## Step 7: Authenticate to Vault and Read a Secret

`vault login -method=cert` presents the SVID as a TLS client certificate. Vault validates it
against the registered Teleport SPIFFE CA and issues a token scoped to the `app-secrets` policy.

The issued token is captured from the login output and passed explicitly to `vault kv get`
so the read runs with the SVID-derived identity — not the admin token from Step 5.

In [ ]:
import json, subprocess

# Authenticate using the SVID as a TLS client certificate
result = subprocess.run(
    [
        'vault', 'login', '-method=cert', '-format=json',
        f'-client-cert={SVID_CERT}',
        f'-client-key={SVID_KEY}',
    ],
    capture_output=True, text=True, env=vault_env,
)
if result.returncode != 0:
    raise RuntimeError(f'vault login failed: {result.stderr}')

login_data  = json.loads(result.stdout)
vault_token = login_data['auth']['client_token']
print(f'Authenticated')
print(f'Token accessor: {login_data["auth"]["accessor"]}')
print(f'Policies      : {login_data["auth"]["policies"]}')
print(f'Lease duration: {login_data["auth"]["lease_duration"]}s')

# Read the secret using the token issued to the SVID identity — not the admin token
svid_env = {**vault_env, 'VAULT_TOKEN': vault_token}
result = subprocess.run(
    ['vault', 'kv', 'get', '-mount=secret', 'myapp/config'],
    capture_output=True, text=True, env=svid_env,
)
if result.returncode != 0:
    raise RuntimeError(f'vault kv get failed: {result.stderr}')
print()
print(result.stdout)


## Step 8 (Optional): Automated Renewal with tbot

Everything above used `tsh workload-identity issue-x509`, which is meant for a human operator
issuing a one-off SVID — useful for exactly what you just did: testing and debugging the
integration interactively. A production workload that needs to stay authenticated indefinitely
shouldn't be re-running `tsh` by hand; that's what `tbot` is for.

`tbot` runs continuously alongside the workload, renewing the SVID before it expires and writing
the updated files to the same output directory every time — the workload doesn't need to change
how it reads `svid.pem` and `svid_key.pem`, only how often it re-authenticates with them.

SVIDs expire after 1 hour. The workload should periodically re-run the `vault login -method=cert`
step from Step 7 (e.g., every 50 minutes) to refresh its Vault token before the SVID expires.

In [ ]:
TBOT_TOKEN = os.environ.get('TBOT_TOKEN', 'your-bot-join-token')

tbot_yaml = f"""version: v2
proxy_server: {TELEPORT_CLUSTER}:443
onboarding:
  join_method: token
  token: {TBOT_TOKEN}
storage:
  type: directory
  path: /opt/machine-id
outputs:
  - type: workload-identity-x509
    selector:
      name: vault-test
    destination:
      type: directory
      path: {SVID_DIR}
"""

outfile = WORK_DIR / 'tbot.yaml'
outfile.write_text(tbot_yaml)
print(tbot_yaml)
print('Start tbot with:')
print(f'  tbot start --config={outfile}')
print()
print('The workload re-authenticates by re-running the Step 7 vault login cell above.')


---
## Troubleshooting

If a cell above raised instead of printing the expected success message, check here before re-running:

| Symptom | Cause |
|:--------|:------|
| `missing name in alias` | The SVID has an empty CN — confirm `subject_template.common_name` is set in Step 3's `WorkloadIdentity`. |
| `failed to match all constraints` | The CA registered with Vault in Step 5c doesn't match the SVID's issuer — re-check Step 4 used `--type=tls-spiffe`, not `tls-host`. |
| `access to workload_identity denied` | The issuer role from Step 3 is missing `workload_identity_labels` or the `workload_identity_token` `use` rule — re-login (`tsh logout --all && tsh login`) after any role change. |
| `vault login` fails with a TLS/handshake error | Vault's listener must not set `tls_disable_client_certs` — cert auth requires mTLS. The appendix's `vault.hcl` already has this unset. |
| Step 7 worked earlier but fails now | SVIDs default to a 1-hour TTL — re-run Step 6 to issue a fresh one, or see Step 8 for automatic renewal via `tbot`. |

---
## Cleanup

Run this cell to remove everything created by the notebook. It is idempotent — safe to run multiple
times or partially (skips resources that do not exist).

In [ ]:
import shutil, subprocess, json, os
from pathlib import Path

# ── Local workspace ───────────────────────────────────────────────────────────
work = Path('./workspace')
if work.exists():
    shutil.rmtree(work)
    print('  removed dir  workspace/')
else:
    print('  skip         workspace/ (not found)')

# ── Vault resources ───────────────────────────────────────────────────────────
if VAULT_TOKEN and VAULT_ADDR:
    v_env = {**os.environ, 'VAULT_ADDR': VAULT_ADDR, 'VAULT_TOKEN': VAULT_TOKEN}
    v_env.pop('VAULT_CACERT', None)
    v_env.pop('VAULT_SKIP_VERIFY', None)
    if VAULT_CACERT:
        v_env['VAULT_CACERT'] = VAULT_CACERT
    else:
        v_env['VAULT_SKIP_VERIFY'] = 'true'

    vault_targets = [
        (['vault', 'delete', 'auth/cert/certs/teleport-svid'],       'cert auth role'),
        (['vault', 'policy', 'delete', 'app-secrets'],                'policy app-secrets'),
        (['vault', 'kv', 'delete', '-mount=secret', 'myapp/config'], 'test secret'),
    ]
    for cmd, label in vault_targets:
        r = subprocess.run(cmd, capture_output=True, text=True, env=v_env)
        status = 'removed' if r.returncode == 0 else 'not found / skipped'
        print(f'  vault {label}: {status}')
else:
    print('  VAULT_TOKEN not set — skipping Vault resource cleanup')

# ── Teleport resources ────────────────────────────────────────────────────────
for cmd, label in [
    (['tctl', 'rm', 'workload_identity/vault-test'], 'workload identity'),
    (['tctl', 'rm', 'role/workload-identity-issuer'], 'issuer role resource'),
]:
    r = subprocess.run(cmd, capture_output=True, text=True)
    status = 'removed' if r.returncode == 0 else 'not found / skipped'
    print(f'  teleport {label}: {status}')

# Remove issuer role assignment from the user (preserve all other roles)
r = subprocess.run(
    ['tctl', 'get', f'users/{TELEPORT_USER}', '--format=json'],
    capture_output=True, text=True,
)
if r.returncode == 0:
    users = json.loads(r.stdout)
    current_roles = users[0]['spec']['roles']
    if 'workload-identity-issuer' in current_roles:
        new_roles = [role for role in current_roles if role != 'workload-identity-issuer']
        r2 = subprocess.run(
            ['tctl', 'users', 'update', TELEPORT_USER, '--set-roles', ','.join(new_roles)],
            capture_output=True, text=True,
        )
        status = 'removed from user' if r2.returncode == 0 else f'error: {r2.stderr.strip()}'
        print(f'  teleport issuer role assignment: {status}')
    else:
        print('  teleport issuer role assignment: not assigned / skipped')
else:
    print('  could not fetch user — skipping role cleanup')

print()
print('Cleanup complete.')
print('To also tear down the Docker Vault: cd workspace/vault-test && docker compose down -v')


---
<a id="appendix-vault-docker-setup"></a>
## Appendix: Vault Setup with Docker

This appendix provisions a single-node Vault instance with TLS using Docker Compose —
matching the setup used to develop this notebook. Skip this section if you already have a Vault instance.

**Additional prerequisites:** Docker and Docker Compose installed.

The cells below:
1. Write `docker-compose.yaml` and `config/vault.hcl` into a `vault-test/` directory
2. Generate a self-signed TLS certificate for the Vault listener
3. Start the container, initialize Vault, and unseal it
4. Enable the KV v2 secrets engine

After completing this section, update `.env` with the root token and CA cert path, then run Step 2
(Configure Environment) again before continuing with the main demo.

In [ ]:
import os, subprocess, json
from pathlib import Path

# IP or hostname the Vault container is reachable at from the notebook host.
# Use 127.0.0.1 for local Docker; set to your VM/host IP for a remote VM.
VAULT_IP   = os.environ.get('VAULT_IP', '127.0.0.1')
VAULT_PORT = 8200
VAULT_DIR  = WORK_DIR / 'vault-test'

for d in ['config', 'data', 'tls']:
    (VAULT_DIR / d).mkdir(parents=True, exist_ok=True)

print(f'Vault IP   : {VAULT_IP}')
print(f'Vault port : {VAULT_PORT}')
print(f'Working dir: {VAULT_DIR.resolve()}')


In [ ]:
docker_compose_yaml = """services:
  vault:
    image: hashicorp/vault:latest
    container_name: vault
    cap_add:
      - IPC_LOCK
    ports:
      - "8200:8200"
      - "8201:8201"
    volumes:
      - ./config:/vault/config
      - ./data:/vault/data
      - ./tls:/vault/tls
    command: server
    restart: unless-stopped
"""

(VAULT_DIR / 'docker-compose.yaml').write_text(docker_compose_yaml)
print(docker_compose_yaml)

In [ ]:
vault_hcl = f"""ui = true

storage "raft" {{
  path    = "/vault/data"
  node_id = "vault-node-1"
}}

listener "tcp" {{
  address       = "0.0.0.0:8200"
  tls_cert_file = "/vault/tls/vault.crt"
  tls_key_file  = "/vault/tls/vault.key"
}}

api_addr     = "https://{VAULT_IP}:{VAULT_PORT}"
cluster_addr = "https://{VAULT_IP}:8201"

disable_mlock = true
"""

(VAULT_DIR / 'config' / 'vault.hcl').write_text(vault_hcl)
print(vault_hcl)

In [ ]:
# Generate a self-signed TLS cert with the Vault IP as a SAN.
# hvac and the vault CLI both validate the SAN, so the IP must be listed explicitly.
result = subprocess.run(
    [
        'openssl', 'req', '-x509', '-newkey', 'rsa:4096',
        '-keyout', str(VAULT_DIR / 'tls' / 'vault.key'),
        '-out',    str(VAULT_DIR / 'tls' / 'vault.crt'),
        '-days', '365', '-nodes',
        '-subj', '/CN=vault',
        '-addext', f'subjectAltName=IP:{VAULT_IP},IP:127.0.0.1,DNS:localhost',
    ],
    capture_output=True, text=True,
)
if result.returncode != 0:
    raise RuntimeError(f'openssl failed: {result.stderr}')

r = subprocess.run(
    ['openssl', 'x509', '-in', str(VAULT_DIR / 'tls' / 'vault.crt'),
     '-noout', '-subject', '-ext', 'subjectAltName'],
    capture_output=True, text=True,
)
print(r.stdout)
print(f'TLS cert written to {VAULT_DIR}/tls/vault.crt')

In [ ]:
import time

result = subprocess.run(
    ['docker', 'compose', 'up', '-d'],
    capture_output=True, text=True, cwd=str(VAULT_DIR),
)
print(result.stdout or result.stderr)
if result.returncode != 0:
    raise RuntimeError(f'docker compose up failed: {result.stderr}')

# Give Vault a moment to start before we talk to it
time.sleep(3)
print('Container started.')

### Initialize and Unseal Vault

Vault starts **sealed**. The `init` step generates unseal keys and the root token.
**Save the output** — the root token cannot be retrieved again.

After the cell completes, copy the printed `VAULT_TOKEN` and `VAULT_CACERT` values into your `.env` file,
then re-run **Step 2** (Configure Environment) so the main demo cells pick them up.

In [ ]:
_vault_admin_addr = f'https://{VAULT_IP}:{VAULT_PORT}'
_vault_env_base   = {**os.environ, 'VAULT_ADDR': _vault_admin_addr, 'VAULT_SKIP_VERIFY': 'true'}

result = subprocess.run(
    ['vault', 'operator', 'init', '-key-shares=3', '-key-threshold=2', '-format=json'],
    capture_output=True, text=True, env=_vault_env_base,
)

if result.returncode != 0:
    if 'already initialized' in result.stderr:
        print('Vault already initialized — set UNSEAL_KEYS and ROOT_TOKEN manually, then run the unseal cell.')
    else:
        raise RuntimeError(f'vault operator init failed: {result.stderr}')
else:
    _init = json.loads(result.stdout)
    UNSEAL_KEYS = _init['unseal_keys_b64']
    ROOT_TOKEN  = _init['root_token']

    print('=== Save these values ===')
    print(f'Root token : {ROOT_TOKEN}')
    for i, k in enumerate(UNSEAL_KEYS, 1):
        print(f'Unseal key {i}: {k}')
    print()
    print('Add to .env:')
    print(f'  VAULT_TOKEN={ROOT_TOKEN}')
    print(f'  VAULT_CACERT=./workspace/vault-test/tls/vault.crt')

In [ ]:
# Unseal with the first 2 keys (threshold=2 of 3 shares)
for key in UNSEAL_KEYS[:2]:
    r = subprocess.run(
        ['vault', 'operator', 'unseal', key],
        capture_output=True, text=True, env=_vault_env_base,
    )
    print(r.stdout.strip())

# Confirm sealed status
r = subprocess.run(
    ['vault', 'status', '-format=json'],
    capture_output=True, text=True, env=_vault_env_base,
)
if r.stdout:
    s = json.loads(r.stdout)
    print(f'Sealed   : {s["sealed"]}')
    print(f'Threshold: {s["t"]}  Shares: {s["n"]}')

In [ ]:
# Enable the KV v2 engine at the 'secret/' path (required by Step 5)
_vault_env_root = {**_vault_env_base, 'VAULT_TOKEN': ROOT_TOKEN}

r = subprocess.run(
    ['vault', 'secrets', 'enable', '-path=secret', 'kv-v2'],
    capture_output=True, text=True, env=_vault_env_root,
)
# 'path is already in use' is not a failure
print(r.stdout or r.stderr)
print()
print('Vault is ready. Update .env with VAULT_TOKEN and VAULT_CACERT, then re-run Step 2.')